In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pip install openpyxl

In [ ]:

xl = pd.ExcelFile("/Users/amankumar/01_ML/AI-Powered Chatbot.xlsx")

print(xl.sheet_names)


In [ ]:
df = pd.read_excel("/Users/amankumar/01_ML/AI-Powered Chatbot.xlsx", sheet_name=0)

In [ ]:
df.head()

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

xl = pd.ExcelFile("/Users/amankumar/01_ML/AI-Powered Chatbot.xlsx")
print(xl.sheet_names)

In [ ]:
df = pd.read_excel("/Users/amankumar/01_ML/AI-Powered Chatbot.xlsx")


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape


In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().sum()/len(df))*100

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
df['Intent'].value_counts()

In [ ]:
df['Intent'].value_counts().plot(kind='bar', figsize=(12,6))

In [ ]:
df['Topic'].value_counts()

In [ ]:
df['Topic'].value_counts().plot(kind='bar', figsize=(12,6))

In [ ]:
df['Sentiment Label'].value_counts()

In [ ]:
df['Sentiment Label'].value_counts().plot(kind='bar')

In [ ]:
df['User Message'].head(10)

In [ ]:
df[['User Message', 'Intent', 'Bot Response']].head(20)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['Intent'].value_counts()

In [ ]:
df['Topic'].value_counts()

In [ ]:
df['Sentiment Label'].value_counts()

In [ ]:
df[['User Message', 'Intent', 'Bot Response']].value_counts().sum()

In [ ]:
df['User Message'].str.len().describe()

In [ ]:
print(df.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Data
X = df["User Message"]
y = df["Intent"]

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# TF-IDF
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Prediction
y_pred = model.predict(X_test_tfidf)

print("✅ Chatbot trained successfully!")
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def chatbot(user_message):

    # User message ko vector me convert karo
    message_vector = vectorizer.transform([user_message])

    # Agar message me dataset ke known words hi nahi hain
    if message_vector.nnz == 0:
        return "unknown", "Sorry, I don't know the answer."

    # Dataset ke saare User Messages ko vector me convert karo
    dataset_vectors = vectorizer.transform(df["User Message"])

    # User question aur dataset ke questions ki similarity
    similarities = cosine_similarity(message_vector, dataset_vectors)[0]

    # Sabse similar question
    max_similarity = similarities.max()

    print("Similarity:", round(max_similarity, 3))

    # Dataset se bahut different hai
    if max_similarity < 0.30:
        return "unknown", "Sorry, I don't know the answer."

    # Intent predict karo
    predicted_intent = model.predict(message_vector)[0]

    # Predicted intent ka response nikalo
    matching_rows = df[df["Intent"] == predicted_intent]

    if len(matching_rows) > 0:
        response = matching_rows.iloc[0]["Bot Response"]
    else:
        return "unknown", "Sorry, I don't know the answer."

    return predicted_intent, response

In [ ]:
while True:

    user_message = input("\nYou: ")

    if user_message.lower() in ["exit", "quit", "bye"]:
        print("Bot: Goodbye! 👋")
        break

    intent, response = chatbot(user_message)

    print("Intent:", intent)
    print("Bot:", response)